# Middleware ordering and `MiddlewareStack`

Two **executed** paths, both using a **real** chat model (`ChatOpenAI`):

| | |
|--|--|
| **Baseline** | You pass `middleware=[...]` to [`create_agent`](https://reference.langchain.com/python/langchain/agents/create_agent) in a **fixed manual order**. |
| **Improved** | You register the **same** middleware instances on a `MiddlewareStack` in **any** order; `resolve()` produces the list LangChain expects (**outermost first**). |

**Prerequisite:** `OPENAI_API_KEY` in the environment or `.env` (loaded above). Without it, the baseline/improved cells **skip** and you can still run the **appendix** (offline demos).

**Ideas in this notebook:** positional lists encode semantics in indices; declarative `slug` + `after` / `before` builds a **DAG** and a stable **topological sort**; optional **`wires`** between layers after resolution.

**Appendix (optional, no API key):** toy `wrap(handler)` chain and LangChain’s `FakeListChatModel` — **not** the primary learning path.


In [ ]:
# Editable install + notebook extras (langchain, jupyter, langchain-openai, …)
import subprocess, sys, shutil
from pathlib import Path

here = Path.cwd()
repo = str(here if (here / "pyproject.toml").exists() else here.parent)
extras = f"{repo}[notebook]"

if shutil.which("uv"):
    cmd = ["uv", "pip", "install", "-q", "-e", extras, "--python", sys.executable]
else:
    cmd = [sys.executable, "-m", "pip", "install", "-q", "-e", extras]

r = subprocess.run(cmd, capture_output=True, text=True)
if r.returncode != 0:
    print("INSTALL FAILED:", r.stderr[:500])
else:
    import importlib.metadata as im
    try:
        v = im.version("langchain-middleware-stack")
    except im.PackageNotFoundError:
        v = "editable"
    print(f"langchain-middleware-stack {v} + notebook extras: ready")


In [ ]:
import os
try:
    from dotenv import load_dotenv, find_dotenv
    env_file = find_dotenv(usecwd=True)
    load_dotenv(env_file, override=False) if env_file else None
    print("dotenv:", env_file or "(none)")
except ImportError:
    print("python-dotenv missing — run make setup")

_api_key = (os.getenv("OPENAI_API_KEY") or "").strip()
# treat empty or obvious placeholder as missing
HAS_OPENAI = bool(_api_key) and _api_key not in ("sk-...", "your-key-here")

def env_on(x):
    return "set" if x else "unset"
print("OPENAI_API_KEY:", env_on(_api_key), "| HAS_OPENAI:", HAS_OPENAI)
print("LANGCHAIN_API_KEY:", env_on(os.getenv("LANGCHAIN_API_KEY")))


In [ ]:
import logging
import signal
import time
from collections.abc import Callable
from typing import Any, ClassVar

logging.basicConfig(level=logging.INFO, format="    %(message)s", force=True)
print("imports OK")


## Baseline — positional `middleware=` (real LLM)

You choose the **exact list order** passed to `create_agent`. Here: **logging → cache → rate limit → retry → timeout** (outermost → innermost), matching the constraints we will encode declaratively later.

Each class below subclasses LangChain’s **`AgentMiddleware`** (so `create_agent` accepts it) and this package’s **`BaseMiddleware`** (so `MiddlewareStack` can sort it). Hooks use **`wrap_model_call`** — the real interception point for model calls.


In [ ]:
from langchain.agents.middleware import AgentMiddleware
from langchain_middleware_stack import BaseMiddleware, MiddlewareStack

mw_log = logging.getLogger("middleware")


class LoggingMiddleware(AgentMiddleware, BaseMiddleware):
    slug: ClassVar[str] = "logging"
    tools: ClassVar[tuple] = ()

    def wrap_model_call(self, request, handler):
        mw_log.info("[logging] start")
        t0 = time.monotonic()
        try:
            resp = handler(request)
            mw_log.info("[logging] end %.0fms", (time.monotonic() - t0) * 1000)
            return resp
        except Exception as e:
            mw_log.error("[logging] err %s", e)
            raise


class CacheMiddleware(AgentMiddleware, BaseMiddleware):
    slug: ClassVar[str] = "cache"
    after: ClassVar[tuple[str, ...]] = ("logging",)
    tools: ClassVar[tuple] = ()

    def __init__(self):
        self._cache: dict[str, Any] = {}
        self.hits = self.misses = 0

    def wrap_model_call(self, request, handler):
        key = repr([(type(m).__name__, getattr(m, "content", "")) for m in request.messages])
        if key in self._cache:
            self.hits += 1
            mw_log.info("[cache] HIT")
            return self._cache[key]
        self.misses += 1
        mw_log.info("[cache] MISS")
        r = handler(request)
        self._cache[key] = r
        return r


class RateLimitMiddleware(AgentMiddleware, BaseMiddleware):
    slug: ClassVar[str] = "rate-limit"
    after: ClassVar[tuple[str, ...]] = ("cache",)
    before: ClassVar[tuple[str, ...]] = ("retry",)
    tools: ClassVar[tuple] = ()

    def __init__(self, max_calls: int = 10, window: float = 1.0):
        self.max_calls, self.window = max_calls, window
        self._calls: list[float] = []

    def wrap_model_call(self, request, handler):
        now = time.monotonic()
        self._calls = [t for t in self._calls if now - t < self.window]
        if len(self._calls) >= self.max_calls:
            raise RuntimeError("rate limit")
        self._calls.append(now)
        mw_log.info("[rl] ok %s/%s", len(self._calls), self.max_calls)
        return handler(request)


class RetryMiddleware(AgentMiddleware, BaseMiddleware):
    slug: ClassVar[str] = "retry"
    after: ClassVar[tuple[str, ...]] = ("cache",)
    tools: ClassVar[tuple] = ()

    def __init__(self, max_retries: int = 2, delay: float = 0.0):
        self.max_retries, self.delay = max_retries, delay
        self.total_attempts = 0

    def wrap_model_call(self, request, handler):
        last = None
        for attempt in range(self.max_retries + 1):
            try:
                self.total_attempts += 1
                return handler(request)
            except Exception as e:
                last = e
                mw_log.warning("[retry] %s", e)
                if attempt < self.max_retries:
                    time.sleep(self.delay)
        raise RuntimeError("exhausted") from last


class TimeoutMiddleware(AgentMiddleware, BaseMiddleware):
    slug: ClassVar[str] = "timeout"
    after: ClassVar[tuple[str, ...]] = ("retry",)
    tools: ClassVar[tuple] = ()

    def __init__(self, seconds: int = 60):
        self.seconds = seconds

    def _h(self, s, f):
        raise TimeoutError(self.seconds)

    def wrap_model_call(self, request, handler):
        mw_log.info("[timeout] %ss", self.seconds)
        old = signal.signal(signal.SIGALRM, self._h)
        signal.alarm(self.seconds)
        try:
            return handler(request)
        finally:
            signal.alarm(0)
            signal.signal(signal.SIGALRM, old)


print("AgentMiddleware + BaseMiddleware demo classes ready")


In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

DEMO_PROMPT = "Reply in one short sentence: what is 2+2?"

if not HAS_OPENAI:
    print("SKIP baseline — set OPENAI_API_KEY to run this cell.")
else:
    model_name = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
    chat = ChatOpenAI(model=model_name, temperature=0, max_tokens=128, api_key=_api_key)

    # Manual order: outermost first (same target order as the stack will resolve to)
    baseline_layers = [
        LoggingMiddleware(),
        CacheMiddleware(),
        RateLimitMiddleware(30, window=5.0),
        RetryMiddleware(2, delay=0.2),
        TimeoutMiddleware(45),
    ]
    print("baseline order (slugs):", [m.slug for m in baseline_layers])

    agent = create_agent(model=chat, tools=[], middleware=baseline_layers)
    out = agent.invoke({"messages": [HumanMessage(DEMO_PROMPT)]})
    print(out["messages"][-1].content)
    c = next(m for m in baseline_layers if m.slug == "cache")
    print("cache hits/misses:", c.hits, c.misses)


## Why the positional list is a bottleneck

- **Semantics live in indices** — “retry outside timeout” vs the opposite is a reorder; nothing on the class states the rule.
- **Merging stacks from two packages** — both need a single owner of the final list and out-of-band agreement.
- **`MiddlewareStack`** lets each class declare **`after` / `before`** on **`slug`** values; `resolve()` validates the graph and runs a stable topological sort (Kahn, tie-break on registration order).


## Improved — same LLM, order from `MiddlewareStack`

Same five middleware **types**, fresh instances, registered in **intentionally scrambled** order. After `resolve()`, `create_agent` receives the same **outermost → innermost** order as the baseline run.


In [ ]:
if not HAS_OPENAI:
    print("SKIP improved — set OPENAI_API_KEY to run this cell.")
else:
    from langchain_core.messages import HumanMessage

    model_name = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
    chat = ChatOpenAI(model=model_name, temperature=0, max_tokens=128, api_key=_api_key)

    m_log = LoggingMiddleware()
    m_cache = CacheMiddleware()
    m_rl = RateLimitMiddleware(30, window=5.0)
    m_retry = RetryMiddleware(2, delay=0.2)
    m_timeout = TimeoutMiddleware(45)

    stack = MiddlewareStack()
    stack.add([m_timeout, m_retry, m_log, m_rl, m_cache])

    resolved = stack.resolve()
    print("resolved order (slugs):", [m.slug for m in resolved])

    agent2 = create_agent(model=chat, tools=[], middleware=resolved)
    out2 = agent2.invoke({"messages": [HumanMessage(DEMO_PROMPT)]})
    print(out2["messages"][-1].content)
    print("cache hits/misses:", m_cache.hits, m_cache.misses, "| retry attempts:", m_retry.total_attempts)


### Constraint graph (Mermaid)

The stack above is rebuilt here only to print the diagram.


In [ ]:
stack_viz = MiddlewareStack()
stack_viz.add(
    [
        TimeoutMiddleware(45),
        RetryMiddleware(2, 0),
        RateLimitMiddleware(30, 5.0),
        CacheMiddleware(),
        LoggingMiddleware(),
    ]
)
print(stack_viz.draw_mermaid())


## Cycle detection

Inconsistent `after` / `before` constraints produce a **cycle**; `resolve()` raises `MiddlewareCycleError`.


In [ ]:
from langchain_middleware_stack import MiddlewareCycleError


class Ping(BaseMiddleware):
    slug: ClassVar[str] = "ping"
    after: ClassVar[tuple[str, ...]] = ("pong",)


class Pong(BaseMiddleware):
    slug: ClassVar[str] = "pong"
    after: ClassVar[tuple[str, ...]] = ("ping",)


try:
    MiddlewareStack().add(Ping()).add(Pong()).resolve()
except MiddlewareCycleError as e:
    print("cycle:", e)


## `wires` — resolve-time injection

`wires` maps an attribute on this middleware to `(source_slug, attr_name)` on another instance after `resolve()`. The source slug must appear in `after`.

Below, `TracingMiddleware` is **not** an `AgentMiddleware` (diagram-only); it only participates in stack resolution.


In [ ]:
class AuthMiddleware(BaseMiddleware):
    slug: ClassVar[str] = "auth"

    def __init__(self):
        self.trace_context = {"user": "alice"}

    def wrap(self, handler, *args, **kwargs):
        return handler(*args, **kwargs)


class TracingMiddleware(BaseMiddleware):
    slug: ClassVar[str] = "tracing"
    after: ClassVar[tuple[str, ...]] = ("auth",)
    wires: ClassVar[dict[str, tuple[str, str]]] = {"_ctx": ("auth", "trace_context")}

    def wrap(self, handler, *args, **kwargs):
        mw_log.info("[trace] user=%s", getattr(self, "_ctx", {}).get("user"))
        return handler(*args, **kwargs)


auth_layer, trace_layer = AuthMiddleware(), TracingMiddleware()
MiddlewareStack().add(trace_layer).add(auth_layer).resolve()
print(trace_layer._ctx)


---

## Appendix A — Offline `wrap(handler)` chain (no API key)

Toy model `StubModel` and plain classes `LogWrap` / `CacheWrap` / … — same **outermost-first** composition idea as LangChain’s model-call wrappers, without HTTP. Useful to **see** how swapping list order changes retries, cache, and timeout semantics.


In [ ]:
class StubModel:
    def __init__(self, name: str = "stub-model", fail_times: int = 0, latency: float = 0.0):
        self.name = name
        self._fail_times = fail_times
        self._call_count = 0
        self._latency = latency

    def call(self, prompt: str) -> str:
        self._call_count += 1
        time.sleep(self._latency)
        if self._call_count <= self._fail_times:
            raise RuntimeError(f"[{self.name}] transient error attempt {self._call_count}")
        return f"[{self.name}] ok: {prompt!r} (#{self._call_count})"


def compose(layers: list[Any], core: Callable[..., Any]) -> Callable[..., Any]:
    fn = core
    for layer in reversed(layers):
        inner = fn

        def make_wrapper(middleware, f):
            def wrapper(*a, **k):
                return middleware.wrap(f, *a, **k)

            return wrapper

        fn = make_wrapper(layer, inner)
    return fn


def run(prompt: str, layers: list[Any], model: StubModel) -> str:
    return compose(layers, model.call)(prompt)


print("StubModel + compose() ready")


In [ ]:
class LogWrap:
    def wrap(self, handler, *args, **kwargs):
        mw_log.info("[logging] start")
        t0 = time.monotonic()
        try:
            r = handler(*args, **kwargs)
            mw_log.info("[logging] end %.0fms", (time.monotonic() - t0) * 1000)
            return r
        except Exception as e:
            mw_log.error("[logging] error %.0fms %s", (time.monotonic() - t0) * 1000, e)
            raise


class CacheWrap:
    def __init__(self):
        self._cache = {}
        self.hits = self.misses = 0

    def wrap(self, handler, *args, **kwargs):
        key = str(args) + str(kwargs)
        if key in self._cache:
            self.hits += 1
            mw_log.info("[cache] HIT")
            return self._cache[key]
        self.misses += 1
        mw_log.info("[cache] MISS")
        r = handler(*args, **kwargs)
        self._cache[key] = r
        return r


class RetryWrap:
    def __init__(self, max_retries: int = 2, delay: float = 0.0):
        self.max_retries = max_retries
        self.delay = delay
        self.total_attempts = 0

    def wrap(self, handler, *args, **kwargs):
        last = None
        for attempt in range(self.max_retries + 1):
            try:
                self.total_attempts += 1
                return handler(*args, **kwargs)
            except RuntimeError as e:
                last = e
                mw_log.warning("[retry] fail attempt %s %s", attempt + 1, e)
                if attempt < self.max_retries:
                    time.sleep(self.delay)
        raise RuntimeError("exhausted") from last


class TimeWrap:
    def __init__(self, seconds: int = 5):
        self.seconds = seconds

    def _h(self, signum, frame):
        raise TimeoutError(f"timeout {self.seconds}s")

    def wrap(self, handler, *args, **kwargs):
        mw_log.info("[timeout] budget=%ss", self.seconds)
        old = signal.signal(signal.SIGALRM, self._h)
        signal.alarm(self.seconds)
        try:
            return handler(*args, **kwargs)
        finally:
            signal.alarm(0)
            signal.signal(signal.SIGALRM, old)


print("LogWrap, CacheWrap, RetryWrap, TimeWrap defined")


### Ordering scenarios (offline)


In [ ]:
from IPython.display import HTML, display as idisplay

_PALETTE = {
    "logging": ("#1967d2", "#e8f0fe"),
    "cache": ("#188038", "#e6f4ea"),
    "retry": ("#b96000", "#fef3e8"),
    "timeout": ("#c5221f", "#fce8e6"),
}
_GREY = ("#5f6368", "#f1f3f4")


def show_stack(slugs: list[str], title: str = "", ok: bool = True) -> None:
    def box(slug, inner_html):
        c, bg = _PALETTE.get(slug, _GREY)
        return (
            f'<div style="border:2px solid {c};background:{bg};border-radius:6px;'
            f'padding:10px 14px;margin:4px 0;">'
            f'<b style="color:{c};font-family:monospace;">{slug}</b>'
            f'<div style="margin-top:6px;">{inner_html}</div></div>'
        )

    core = '<div style="border:2px dashed #9aa0a6;padding:8px;font-family:monospace;font-size:12px;">model_call</div>'
    for s in reversed(slugs):
        core = box(s, core)
    color = "#188038" if ok else "#c5221f"
    icon = "OK" if ok else "X"
    head = f'<div style="font-weight:700;color:{color};margin-bottom:8px;font-family:monospace;">[{icon}] {title}</div>' if title else ""
    idisplay(HTML(f'<div style="max-width:640px;font-family:sans-serif;">{head}{core}</div>'))


print("show_stack() ready")


In [ ]:
show_stack(["logging", "cache", "retry", "timeout"], "logging outermost; timeout innermost (per attempt)", ok=True)
model = StubModel("stub-model", fail_times=1)
log_w, cache_w, retry_w, tout_w = LogWrap(), CacheWrap(), RetryWrap(2, 0), TimeWrap(5)
run("2+2?", [log_w, cache_w, retry_w, tout_w], model)
print("cache", cache_w.hits, cache_w.misses, "retry_attempts", retry_w.total_attempts)


In [ ]:
show_stack(["timeout", "retry", "cache", "logging"], "timeout wraps full retry loop", ok=False)
model = StubModel("stub-model", fail_times=1)
log_w, cache_w, retry_w, tout_w = LogWrap(), CacheWrap(), RetryWrap(2, 0), TimeWrap(5)
run("2+2?", [tout_w, retry_w, cache_w, log_w], model)
print("(With a slow real model + many retries, this ordering can cut attempts short.)")


In [ ]:
show_stack(["logging", "retry", "timeout", "cache"], "cache inside retry", ok=False)
model = StubModel("stub-model", fail_times=2)
log_w, cache_w, retry_w, tout_w = LogWrap(), CacheWrap(), RetryWrap(3, 0), TimeWrap(5)
run("2+2?", [log_w, retry_w, tout_w, cache_w], model)
print("cache lookups (misses)", cache_w.misses)


## Appendix B — `FakeListChatModel` (optional)

LangChain’s test double returns **fixed** strings and never calls an API. Use it only if you want to exercise `create_agent` **without** `OPENAI_API_KEY` — it is **not** representative of production behavior.


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRetryMiddleware
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.messages import HumanMessage

stub_chat = FakeListChatModel(responses=["stub response"])
agent_fake = create_agent(
    model=stub_chat,
    tools=[],
    middleware=[
        ModelRetryMiddleware(
            max_retries=1,
            initial_delay=0.0,
            backoff_factor=0.0,
            jitter=False,
        ),
    ],
)
out = agent_fake.invoke({"messages": [HumanMessage("ping")]})
print(out["messages"][-1].content)


## References

- [LangChain — Prebuilt middleware](https://docs.langchain.com/oss/python/langchain/middleware/built-in)
- [LangChain — `create_agent`](https://reference.langchain.com/python/langchain/agents/create_agent)

**This package:** `MiddlewareStack`, `BaseMiddleware`, `slug`, `after`, `before`, `wires`.
